In [3]:
import pandas as pd
import numpy as np
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

# ─── LOAD DATA ────────────────────────────────────────────────────────────────
df26_raw = pd.read_csv('tn_2026_results.csv')
df21_raw = pd.read_csv('tn_2021_results.csv')
electors = pd.read_csv('tn_2026_electors.csv')
master   = pd.read_csv('constituency_master.csv')

In [4]:
df26 = df26_raw.copy()
df21 = df21_raw.copy()

# ─── TOP-3 PARTIES (by total votes) ───────────────────────────────────────────
TOP3_26 = ['TVK', 'DMK', 'AIADMK']
TOP3_21 = ['DMK', 'AIADMK', 'BJP']    # 2021 top 3 by seats / vote share

# ─── COMPUTE 2026 TURNOUT ─────────────────────────────────────────────────────
# electors has total registered voters per constituency
electors_clean = electors[['ac_number','total']].dropna(subset=['ac_number'])
electors_clean['ac_number'] = electors_clean['ac_number'].astype(int)

# total votes cast per constituency in 2026 = sum of all candidate votes incl NOTA
votes_cast_26 = (df26.groupby('ac_number')['votes']
                      .sum()
                      .reset_index()
                      .rename(columns={'votes':'total_votes_cast'}))
votes_cast_26 = votes_cast_26.merge(electors_clean, on='ac_number', how='left')
votes_cast_26['turnout_26'] = (votes_cast_26['total_votes_cast'] / votes_cast_26['total'] * 100).round(2)

# merge turnout back to df26
df26 = df26.merge(votes_cast_26[['ac_number','turnout_26']], on='ac_number', how='left')

# total votes cast per constituency 2021 (derive from turnout col which is constituency-level)
votes_cast_21 = (df21.groupby(['ac_number','constituency','region','reserved'])
                      .agg(total_votes_cast=('votes','sum'), turnout_21=('turnout','first'))
                      .reset_index())

In [5]:
# ─── WINNER DERIVATION ────────────────────────────────────────────────────────
def get_winners(df, turnout_col=None):
    """Return one row per constituency: winner candidate, party, votes, margin."""
    df = df.copy()
    df['rank'] = df.groupby('ac_number')['votes'].rank(method='first', ascending=False)
    winners = df[df['rank'] == 1][['ac_number','constituency','region','reserved','candidate','party','votes']].copy()
    runners = df[df['rank'] == 2][['ac_number','votes']].rename(columns={'votes':'runner_votes'})
    winners = winners.merge(runners, on='ac_number', how='left')
    winners['margin'] = winners['votes'] - winners['runner_votes']
    # total valid votes per constituency
    totals = df.groupby('ac_number')['votes'].sum().reset_index().rename(columns={'votes':'total_valid'})
    winners = winners.merge(totals, on='ac_number', how='left')
    winners['win_pct'] = (winners['votes'] / winners['total_valid'] * 100).round(2)
    if turnout_col:
        t = df[['ac_number', turnout_col]].drop_duplicates()
        winners = winners.merge(t, on='ac_number', how='left')
    return winners

winners26 = get_winners(df26, 'turnout_26')
winners21 = get_winners(df21, 'turnout')

# merge region/reserved into votes_cast_21 if not present
votes_cast_21 = votes_cast_21.merge(
    master[['ac_number','region','reserved']], on='ac_number', how='left'
)

def pprint(title, df, idx=False):
    print(f"\n{'═'*70}")
    print(f"  {title}")
    print(f"{'═'*70}")
    print(tabulate(df, headers='keys', tablefmt='rounded_outline',
                   showindex=idx, floatfmt='.2f'))
    

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 2 — Top & Bottom 5 individual candidate vote shares by party
# ══════════════════════════════════════════════════════════════════════════════
def cand_vote_share(df, year, turnout_col):
    totals = df.groupby('ac_number')['votes'].sum().reset_index().rename(columns={'votes':'total_valid'})
    d = df.merge(totals, on='ac_number')
    d['vote_pct'] = (d['votes'] / d['total_valid'] * 100).round(2)
    d = d[d['party'].isin(TOP3_26 if year == 2026 else ['DMK','AIADMK','BJP','INC','NTK'])]
    d = d[~d['candidate'].str.upper().str.contains('NOTA', na=False)]
    return d[['candidate','party','constituency','votes','vote_pct']]

cand26 = cand_vote_share(df26, 2026, 'turnout_26')
cand21 = cand_vote_share(df21, 2021, 'turnout')

def top_bottom5(df, year):
    rows = []
    parties = df['party'].unique()
    for p in sorted(parties):
        sub = df[df['party'] == p].sort_values('vote_pct', ascending=False)
        top5 = sub.head(5).assign(rank_type='Top 5')
        bot5 = sub.tail(5).assign(rank_type='Bottom 5')
        rows.append(pd.concat([top5, bot5]))
    return pd.concat(rows, ignore_index=True)

tb26 = top_bottom5(cand26, 2026)[['party','rank_type','candidate','constituency','votes','vote_pct']]
tb21 = top_bottom5(cand21, 2021)[['party','rank_type','candidate','constituency','votes','vote_pct']]
tb26.columns = ['Party','Rank Type','Candidate','Constituency','Votes','Vote Share %']
tb21.columns = ['Party','Rank Type','Candidate','Constituency','Votes','Vote Share %']

pprint("METRIC 2 — Top & Bottom 5 Candidate Vote Shares — 2026 (Top 3 Parties)", tb26)
pprint("METRIC 2 — Top & Bottom 5 Candidate Vote Shares — 2021 (Key Parties)", tb21)


══════════════════════════════════════════════════════════════════════
  METRIC 2 — Top & Bottom 5 Candidate Vote Shares — 2026 (Top 3 Parties)
══════════════════════════════════════════════════════════════════════
╭─────────┬─────────────┬──────────────────────────┬────────────────────────┬─────────┬────────────────╮
│ Party   │ Rank Type   │ Candidate                │ Constituency           │   Votes │   Vote Share % │
├─────────┼─────────────┼──────────────────────────┼────────────────────────┼─────────┼────────────────┤
│ AIADMK  │ Top 5       │ EDAPPADI PALANISWAMI. K  │ Edappadi               │  148933 │          57.67 │
│ AIADMK  │ Top 5       │ VIJAYABASKAR. C          │ Viralimalai            │  105773 │          51.87 │
│ AIADMK  │ Top 5       │ S.M.SUKUMAR              │ Arcot                  │  105608 │          46.77 │
│ AIADMK  │ Top 5       │ ANBALAGAN. K.P.          │ Palacodu               │  102807 │          45.64 │
│ AIADMK  │ Top 5       │ KAMARAJ. R             

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 3 — Region-wise vote share of top 3 parties (2021 & 2026)
# ══════════════════════════════════════════════════════════════════════════════
def region_voteshare(df, parties, year):
    totals = df.groupby('region')['votes'].sum().rename('region_total')
    party_votes = df[df['party'].isin(parties)].groupby(['region','party'])['votes'].sum()
    result = (party_votes / totals * 100).round(2).reset_index()
    result.columns = ['Region','Party',f'{year} Vote Share %']
    return result.pivot(index='Region', columns='Party', values=f'{year} Vote Share %').reset_index()

rv26 = region_voteshare(df26, TOP3_26, 2026)
rv21 = region_voteshare(df21, ['DMK','AIADMK','NTK'], 2021)

pprint("METRIC 3 — Region-wise Vote Share of Top 3 Parties — 2026", rv26)
pprint("METRIC 3 — Region-wise Vote Share of Top 3 Parties — 2021", rv21)


══════════════════════════════════════════════════════════════════════
  METRIC 3 — Region-wise Vote Share of Top 3 Parties — 2026
══════════════════════════════════════════════════════════════════════
╭───────────────┬──────────┬───────┬───────╮
│ Region        │   AIADMK │   DMK │   TVK │
├───────────────┼──────────┼───────┼───────┤
│ Central       │    27.00 │ 23.18 │ 31.02 │
│ Chennai Metro │    15.94 │ 24.31 │ 46.63 │
│ Delta         │    19.54 │ 27.66 │ 32.87 │
│ Kongu         │    23.86 │ 25.62 │ 34.49 │
│ North         │    24.93 │ 20.50 │ 33.06 │
│ South         │    16.81 │ 24.64 │ 33.52 │
╰───────────────┴──────────┴───────┴───────╯

══════════════════════════════════════════════════════════════════════
  METRIC 3 — Region-wise Vote Share of Top 3 Parties — 2021
══════════════════════════════════════════════════════════════════════
╭───────────────┬──────────┬───────┬───────╮
│ Region        │   AIADMK │   DMK │   NTK │
├───────────────┼──────────┼───────┼───────┤
│ Central

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 4 — State-wide vote share of parties (2021 & 2026)
# ══════════════════════════════════════════════════════════════════════════════
def state_voteshare(df, year, top_n=12):
    total = df['votes'].sum()
    vs = (df.groupby('party')['votes'].sum() / total * 100).round(2).reset_index()
    vs.columns = ['Party', f'{year} Vote Share %']
    vs = vs[~vs['Party'].isin(['NOTA'])].sort_values(f'{year} Vote Share %', ascending=False).head(top_n)
    return vs

sv26 = state_voteshare(df26, 2026)
sv21 = state_voteshare(df21, 2021)
sv_combined = sv26.merge(sv21, on='Party', how='outer').sort_values('2026 Vote Share %', ascending=False)
sv_combined = sv_combined.fillna('-')

pprint("METRIC 4 — State-wide Party Vote Share (2021 vs 2026)", sv_combined)


══════════════════════════════════════════════════════════════════════
  METRIC 4 — State-wide Party Vote Share (2021 vs 2026)
══════════════════════════════════════════════════════════════════════
╭─────────┬─────────────────────┬─────────────────────╮
│ Party   │ 2026 Vote Share %   │ 2021 Vote Share %   │
├─────────┼─────────────────────┼─────────────────────┤
│ TVK     │ 34.92               │ -                   │
│ DMK     │ 24.19               │ 37.7                │
│ AIADMK  │ 21.21               │ 33.29               │
│ NTK     │ 4.0                 │ 6.58                │
│ INC     │ 3.37                │ 4.27                │
│ BJP     │ 2.97                │ 2.62                │
│ PMK     │ 2.17                │ 3.8                 │
│ DMDK    │ 1.2                 │ -                   │
│ VCK     │ 1.09                │ 0.99                │
│ IND     │ 1.06                │ 1.39                │
│ AMMK    │ 0.86                │ 2.35                │
│ CPI     │ 0.66 

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 6 — Region-wise Won Seats of Top 3 Parties (2021 & 2026)
# ══════════════════════════════════════════════════════════════════════════════
def region_seats(winners, parties, year):
    w = winners[winners['party'].isin(parties)]
    tbl = w.groupby(['region','party']).size().reset_index(name=f'{year} Seats')
    pivot = tbl.pivot(index='region', columns='party', values=f'{year} Seats').fillna(0).astype(int).reset_index()
    pivot.columns.name = None
    # add total seats per region
    total_per_region = winners.groupby('region').size().rename('Total Seats')
    pivot = pivot.merge(total_per_region, on='region')
    return pivot

rs26 = region_seats(winners26, TOP3_26, 2026)
rs21 = region_seats(winners21, ['DMK','AIADMK','INC'], 2021)
rs21_full = region_seats(winners21, winners21['party'].unique(), 2021)

pprint("METRIC 6 — Region-wise Seats Won by Top 3 Parties — 2026", rs26)
pprint("METRIC 6 — Region-wise Seats Won by Top 3 Parties — 2021", rs21)


══════════════════════════════════════════════════════════════════════
  METRIC 6 — Region-wise Seats Won by Top 3 Parties — 2026
══════════════════════════════════════════════════════════════════════
╭───────────────┬──────────┬───────┬───────┬───────────────╮
│ region        │   AIADMK │   DMK │   TVK │   Total Seats │
├───────────────┼──────────┼───────┼───────┼───────────────┤
│ Central       │       15 │     8 │    12 │            41 │
│ Chennai Metro │        1 │     2 │    29 │            32 │
│ Delta         │        4 │    14 │    10 │            33 │
│ Kongu         │        7 │     9 │    16 │            33 │
│ North         │       15 │     4 │    15 │            37 │
│ South         │        5 │    22 │    26 │            58 │
╰───────────────┴──────────┴───────┴───────┴───────────────╯

══════════════════════════════════════════════════════════════════════
  METRIC 6 — Region-wise Seats Won by Top 3 Parties — 2021
═════════════════════════════════════════════════════════

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 7 — Flipped Seats
# ══════════════════════════════════════════════════════════════════════════════
merged_winners = winners21[['ac_number','constituency','party','region']].rename(
    columns={'party':'party_21','constituency':'constituency_21'}).merge(
    winners26[['ac_number','party','constituency']].rename(
        columns={'party':'party_26','constituency':'constituency_26'}),
    on='ac_number', how='inner'
)
merged_winners['flipped'] = merged_winners['party_21'] != merged_winners['party_26']
flipped = merged_winners[merged_winners['flipped']].copy()

# Summary table: lost party, seats lost, captured by TVK, captured by others
def flip_summary(flipped):
    rows = []
    for lost_party in flipped['party_21'].unique():
        sub = flipped[flipped['party_21'] == lost_party]
        seats_lost = len(sub)
        by_tvk = len(sub[sub['party_26'] == 'TVK'])
        by_dmk = len(sub[sub['party_26'] == 'DMK'])
        by_aiadmk = len(sub[sub['party_26'] == 'AIADMK'])
        by_others = seats_lost - by_tvk - by_dmk - by_aiadmk
        rows.append({
            'Lost Party': lost_party,
            'Seats Lost': seats_lost,
            'Captured by TVK': by_tvk,
            'Captured by DMK': by_dmk,
            'Captured by AIADMK': by_aiadmk,
            'Captured by Others': by_others
        })
    df = pd.DataFrame(rows).sort_values('Seats Lost', ascending=False)
    return df[df['Seats Lost'] > 0]

flip_tbl = flip_summary(flipped)
pprint("METRIC 7 — Flipped Seats Summary", flip_tbl)


══════════════════════════════════════════════════════════════════════
  METRIC 7 — Flipped Seats Summary
══════════════════════════════════════════════════════════════════════
╭──────────────┬──────────────┬───────────────────┬───────────────────┬──────────────────────┬──────────────────────╮
│ Lost Party   │   Seats Lost │   Captured by TVK │   Captured by DMK │   Captured by AIADMK │   Captured by Others │
├──────────────┼──────────────┼───────────────────┼───────────────────┼──────────────────────┼──────────────────────┤
│ DMK          │           93 │                65 │                 0 │                   22 │                    6 │
│ AIADMK       │           44 │                26 │                15 │                    0 │                    3 │
│ INC          │           14 │                11 │                 1 │                    0 │                    2 │
│ PMK          │            4 │                 2 │                 0 │                    2 │                    

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 8 — Net Seats Lost/Gained per Party (2021 vs 2026)
# ══════════════════════════════════════════════════════════════════════════════
seats21 = winners21.groupby('party').size().rename('Seats 2021')
seats26 = winners26.groupby('party').size().rename('Seats 2026')
net = pd.concat([seats21, seats26], axis=1).fillna(0).astype(int)
net['Net Change'] = net['Seats 2026'] - net['Seats 2021']
net = net[net[['Seats 2021','Seats 2026']].sum(axis=1) > 0].sort_values('Seats 2021', ascending=False)
net = net.reset_index().rename(columns={'party':'Party'})
pprint("METRIC 8 — Net Seats Won/Lost by Party (2021 → 2026)", net)


══════════════════════════════════════════════════════════════════════
  METRIC 8 — Net Seats Won/Lost by Party (2021 → 2026)
══════════════════════════════════════════════════════════════════════
╭─────────┬──────────────┬──────────────┬──────────────╮
│ Party   │   Seats 2021 │   Seats 2026 │   Net Change │
├─────────┼──────────────┼──────────────┼──────────────┤
│ DMK     │          133 │           59 │          -74 │
│ AIADMK  │           66 │           47 │          -19 │
│ INC     │           18 │            5 │          -13 │
│ PMK     │            5 │            4 │           -1 │
│ BJP     │            4 │            1 │           -3 │
│ VCK     │            4 │            2 │           -2 │
│ CPI     │            2 │            2 │            0 │
│ CPI(M)  │            2 │            2 │            0 │
│ AMMK    │            0 │            1 │            1 │
│ DMDK    │            0 │            1 │            1 │
│ IUML    │            0 │            2 │            2 │
│ TV

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 9 — Average State Turnout (2021 & 2026)
# ══════════════════════════════════════════════════════════════════════════════
avg_to_21 = df21.groupby('ac_number')['turnout'].first().mean()
avg_to_26 = votes_cast_26['turnout_26'].mean()
to_state = pd.DataFrame({
    'Metric': ['Average State Turnout'],
    '2021 (%)': [round(avg_to_21, 2)],
    '2026 (%)': [round(avg_to_26, 2)],
    'Increase (pp)': [round(avg_to_26 - avg_to_21, 2)]
})
pprint("METRIC 9 — Average State Turnout (2021 vs 2026)", to_state)


══════════════════════════════════════════════════════════════════════
  METRIC 9 — Average State Turnout (2021 vs 2026)
══════════════════════════════════════════════════════════════════════
╭───────────────────────┬────────────┬────────────┬─────────────────╮
│ Metric                │   2021 (%) │   2026 (%) │   Increase (pp) │
├───────────────────────┼────────────┼────────────┼─────────────────┤
│ Average State Turnout │      73.37 │      86.07 │           12.70 │
╰───────────────────────┴────────────┴────────────┴─────────────────╯


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 10 — Region-wise Average Turnout (2021 & 2026)
# ══════════════════════════════════════════════════════════════════════════════
# 2021 region turnout — one turnout value per constituency
region_to21 = (df21.groupby(['ac_number','region'])['turnout'].first()
               .reset_index().groupby('region')['turnout'].mean().round(2).rename('Avg Turnout 2021 (%)'))

# 2026 region turnout
df26_to = votes_cast_26[['ac_number','turnout_26']].merge(
    master[['ac_number','region']], on='ac_number', how='left')
region_to26 = df26_to.groupby('region')['turnout_26'].mean().round(2).rename('Avg Turnout 2026 (%)')

region_to = pd.concat([region_to21, region_to26], axis=1).reset_index()
region_to.columns = ['Region','Avg Turnout 2021 (%)','Avg Turnout 2026 (%)']
region_to['Increase (pp)'] = (region_to['Avg Turnout 2026 (%)'] - region_to['Avg Turnout 2021 (%)']).round(2)
region_to['Increase (%)'] = ((region_to['Increase (pp)'] / region_to['Avg Turnout 2021 (%)']) * 100).round(2)
pprint("METRIC 10 — Region-wise Average Turnout (2021 vs 2026)", region_to)


══════════════════════════════════════════════════════════════════════
  METRIC 10 — Region-wise Average Turnout (2021 vs 2026)
══════════════════════════════════════════════════════════════════════
╭───────────────┬────────────────────────┬────────────────────────┬─────────────────┬────────────────╮
│ Region        │   Avg Turnout 2021 (%) │   Avg Turnout 2026 (%) │   Increase (pp) │   Increase (%) │
├───────────────┼────────────────────────┼────────────────────────┼─────────────────┼────────────────┤
│ Central       │                  78.92 │                  89.53 │           10.61 │          13.44 │
│ Chennai Metro │                  63.41 │                  84.60 │           21.19 │          33.42 │
│ Delta         │                  74.91 │                  84.33 │            9.42 │          12.58 │
│ Kongu         │                  73.06 │                  88.24 │           15.18 │          20.78 │
│ North         │                  78.05 │                  89.73 │           1

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 11 — Top & Bottom 5 Turnout Constituencies (2021 & 2026)
# ══════════════════════════════════════════════════════════════════════════════
to21_c = df21.groupby(['ac_number','constituency','region'])['turnout'].first().reset_index()
to21_c.columns = ['ac_number','Constituency','Region','Turnout 2021 (%)']

to26_c = votes_cast_26[['ac_number','turnout_26']].merge(
    master[['ac_number','constituency','region']], on='ac_number', how='left')
to26_c.columns = ['ac_number','Turnout 2026 (%)','Constituency','Region']

top5_to21 = to21_c.nlargest(5,'Turnout 2021 (%)')[['Constituency','Region','Turnout 2021 (%)']].assign(Rank='Top 5')
bot5_to21 = to21_c.nsmallest(5,'Turnout 2021 (%)')[['Constituency','Region','Turnout 2021 (%)']].assign(Rank='Bottom 5')
top5_to26 = to26_c.nlargest(5,'Turnout 2026 (%)')[['Constituency','Region','Turnout 2026 (%)']].assign(Rank='Top 5')
bot5_to26 = to26_c.nsmallest(5,'Turnout 2026 (%)')[['Constituency','Region','Turnout 2026 (%)']].assign(Rank='Bottom 5')

pprint("METRIC 11 — Top 5 Turnout Constituencies 2021", top5_to21)
pprint("METRIC 11 — Bottom 5 Turnout Constituencies 2021", bot5_to21)
pprint("METRIC 11 — Top 5 Turnout Constituencies 2026", top5_to26)
pprint("METRIC 11 — Bottom 5 Turnout Constituencies 2026", bot5_to26)


══════════════════════════════════════════════════════════════════════
  METRIC 11 — Top 5 Turnout Constituencies 2021
══════════════════════════════════════════════════════════════════════
╭────────────────┬──────────┬────────────────────┬────────╮
│ Constituency   │ Region   │   Turnout 2021 (%) │ Rank   │
├────────────────┼──────────┼────────────────────┼────────┤
│ Palacode       │ North    │              87.37 │ Top 5  │
│ Kulithalai     │ Kongu    │              86.16 │ Top 5  │
│ Edapadi        │ Central  │              85.64 │ Top 5  │
│ Veerapandi     │ Central  │              85.64 │ Top 5  │
│ Viralimalai    │ Delta    │              85.43 │ Top 5  │
╰────────────────┴──────────┴────────────────────┴────────╯

══════════════════════════════════════════════════════════════════════
  METRIC 11 — Bottom 5 Turnout Constituencies 2021
══════════════════════════════════════════════════════════════════════
╭────────────────────┬───────────────┬────────────────────┬──────────╮
│ Co

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 14 — Seats Won by Party in SC & ST Reserved Constituencies (2021 & 2026)
# ══════════════════════════════════════════════════════════════════════════════
def reserved_seats(winners, year):
    w = winners[winners['reserved'].isin(['SC','ST'])].copy()
    tbl = w.groupby(['party','reserved']).size().reset_index(name='seats')
    pivot = tbl.pivot(index='party', columns='reserved', values='seats').fillna(0).astype(int)
    if 'SC' not in pivot.columns: pivot['SC'] = 0
    if 'ST' not in pivot.columns: pivot['ST'] = 0
    pivot['Total Reserved'] = pivot['SC'] + pivot['ST']
    pivot = pivot.reset_index().rename(columns={'party':'Party'})
    pivot = pivot[pivot['Total Reserved'] > 0].sort_values('Total Reserved', ascending=False)
    pivot.columns.name = None
    return pivot

res26 = reserved_seats(winners26, 2026)
res21 = reserved_seats(winners21, 2021)
pprint("METRIC 14 — Party Seats in SC/ST Reserved Constituencies — 2026", res26)
pprint("METRIC 14 — Party Seats in SC/ST Reserved Constituencies — 2021", res21)


══════════════════════════════════════════════════════════════════════
  METRIC 14 — Party Seats in SC/ST Reserved Constituencies — 2026
══════════════════════════════════════════════════════════════════════
╭─────────┬──────┬──────┬──────────────────╮
│ Party   │   SC │   ST │   Total Reserved │
├─────────┼──────┼──────┼──────────────────┤
│ TVK     │   23 │    1 │               24 │
│ AIADMK  │    8 │    1 │                9 │
│ DMK     │    9 │    0 │                9 │
│ VCK     │    2 │    0 │                2 │
│ CPI     │    1 │    0 │                1 │
│ CPI(M)  │    1 │    0 │                1 │
╰─────────┴──────┴──────┴──────────────────╯

══════════════════════════════════════════════════════════════════════
  METRIC 14 — Party Seats in SC/ST Reserved Constituencies — 2021
══════════════════════════════════════════════════════════════════════
╭─────────┬──────┬──────┬──────────────────╮
│ Party   │   SC │   ST │   Total Reserved │
├─────────┼──────┼──────┼─────────────────

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 15 — Reserved Seats Turnout Stats (SC & ST)
# ══════════════════════════════════════════════════════════════════════════════
res_to = master[['ac_number','reserved']].merge(
    votes_cast_26[['ac_number','turnout_26']], on='ac_number').merge(
    df21.groupby('ac_number')['turnout'].first().reset_index().rename(columns={'turnout':'turnout_21'}),
    on='ac_number', how='left')

def reserved_turnout_stats(res_to):
    rows = []
    for cat in ['SC','ST','GEN']:
        sub = res_to[res_to['reserved'] == cat]
        rows.append({
            'Category': cat,
            'Count': len(sub),
            'Avg Turnout 2021 (%)': round(sub['turnout_21'].mean(), 2),
            'Avg Turnout 2026 (%)': round(sub['turnout_26'].mean(), 2),
            'Min Turnout 2026 (%)': round(sub['turnout_26'].min(), 2),
            'Max Turnout 2026 (%)': round(sub['turnout_26'].max(), 2),
        })
    return pd.DataFrame(rows)

res_stats = reserved_turnout_stats(res_to)
pprint("METRIC 15 — Turnout Stats by Reservation Category (2021 & 2026)", res_stats)


══════════════════════════════════════════════════════════════════════
  METRIC 15 — Turnout Stats by Reservation Category (2021 & 2026)
══════════════════════════════════════════════════════════════════════
╭────────────┬─────────┬────────────────────────┬────────────────────────┬────────────────────────┬────────────────────────╮
│ Category   │   Count │   Avg Turnout 2021 (%) │   Avg Turnout 2026 (%) │   Min Turnout 2026 (%) │   Max Turnout 2026 (%) │
├────────────┼─────────┼────────────────────────┼────────────────────────┼────────────────────────┼────────────────────────┤
│ SC         │      44 │                  75.26 │                  86.95 │                  79.22 │                  93.24 │
│ ST         │       2 │                  82.01 │                  92.14 │                  91.37 │                  92.91 │
│ GEN        │     188 │                  72.84 │                  85.80 │                  70.14 │                  94.25 │
╰────────────┴─────────┴─────────────────

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 16 — 10 Minimum Margin Seats (2026)
# ══════════════════════════════════════════════════════════════════════════════
def build_margin_table(df, year, n=10, ascending=True):
    df = df.copy()
    totals = df.groupby('ac_number')['votes'].sum().rename('total_valid')
    df['rank'] = df.groupby('ac_number')['votes'].rank(method='first', ascending=False)
    winner = df[df['rank'] == 1][['ac_number','constituency','region','candidate','party','votes']].rename(
        columns={'candidate':'Winner','party':'Win Party','votes':'Winner Votes'})
    runner = df[df['rank'] == 2][['ac_number','candidate','party','votes']].rename(
        columns={'candidate':'Runner-Up','party':'Runner Party','votes':'Runner Votes'})
    tbl = winner.merge(runner, on='ac_number').merge(totals, on='ac_number')
    tbl['Margin'] = tbl['Winner Votes'] - tbl['Runner Votes']
    tbl = tbl.sort_values('Margin', ascending=ascending).head(n)
    tbl = tbl[['constituency','region','Winner','Win Party','Winner Votes','Runner-Up','Runner Party','Runner Votes','Margin']]
    tbl.columns = ['Constituency','Region','Winner','Win Party','Winner Votes','Runner-Up','Runner Party','Runner Votes','Margin']
    return tbl

min_margin_26 = build_margin_table(df26, 2026, n=10, ascending=True)
pprint("METRIC 16 — 10 Minimum Margin Seats 2026", min_margin_26)


══════════════════════════════════════════════════════════════════════
  METRIC 16 — 10 Minimum Margin Seats 2026
══════════════════════════════════════════════════════════════════════
╭─────────────────┬──────────┬─────────────────────────┬─────────────┬────────────────┬───────────────────────┬────────────────┬────────────────┬──────────╮
│ Constituency    │ Region   │ Winner                  │ Win Party   │   Winner Votes │ Runner-Up             │ Runner Party   │   Runner Votes │   Margin │
├─────────────────┼──────────┼─────────────────────────┼─────────────┼────────────────┼───────────────────────┼────────────────┼────────────────┼──────────┤
│ Tiruppattur S.  │ South    │ SEENIVASA SETHUPATHY. R │ TVK         │          83375 │ PERIAKARUPPAN. KR     │ DMK            │          83374 │        1 │
│ Veppanahalli    │ North    │ SRINIVASAN.P.S          │ DMK         │          74691 │ MUNUSAMY.K.P          │ AIADMK         │          74553 │      138 │
│ Kanniyakumari   │ South    

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 17 — 10 Maximum Margin Seats (2026)
# ══════════════════════════════════════════════════════════════════════════════
max_margin_26 = build_margin_table(df26, 2026, n=10, ascending=False)
pprint("METRIC 17 — 10 Maximum Margin Seats 2026", max_margin_26)


══════════════════════════════════════════════════════════════════════
  METRIC 17 — 10 Maximum Margin Seats 2026
══════════════════════════════════════════════════════════════════════
╭──────────────────┬───────────────┬─────────────────────────┬─────────────┬────────────────┬───────────────────┬────────────────┬────────────────┬──────────╮
│ Constituency     │ Region        │ Winner                  │ Win Party   │   Winner Votes │ Runner-Up         │ Runner Party   │   Runner Votes │   Margin │
├──────────────────┼───────────────┼─────────────────────────┼─────────────┼────────────────┼───────────────────┼────────────────┼────────────────┼──────────┤
│ Edappadi         │ Central       │ EDAPPADI PALANISWAMI. K │ AIADMK      │         148933 │ PREMKUMAR. K      │ IND            │          50823 │    98110 │
│ Shozhinganallur  │ Chennai Metro │ ECR P SARAVANAN         │ TVK         │         220382 │ S. ARAVIND RAMESH │ DMK            │         123602 │    96780 │
│ Madavaram        

In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 18 — Candidates with >50% Vote Share in their Constituency
# ══════════════════════════════════════════════════════════════════════════════
def over50(df, year):
    totals = df.groupby('ac_number')['votes'].sum().rename('total_valid')
    d = df.merge(totals, on='ac_number')
    d['pct'] = d['votes'] / d['total_valid'] * 100
    over = d[d['pct'] > 50]
    return over[['constituency','candidate','party','votes','pct']].rename(
        columns={'constituency':'Constituency','candidate':'Candidate',
                 'party':'Party','votes':'Votes','pct':'Vote Share %'}).sort_values('Vote Share %', ascending=False)

o50_26 = over50(df26, 2026)
o50_21 = over50(df21, 2021)
print(f"\n{'═'*70}")
print(f"  METRIC 18 — Candidates with >50% Vote Share")
print(f"{'═'*70}")
print(f"  2021: {len(o50_21)} candidates crossed 50%")
print(f"  2026: {len(o50_26)} candidates crossed 50%")
pprint("METRIC 18 — All Candidates >50% Vote Share — 2021", o50_21.head(20))
pprint("METRIC 18 — All Candidates >50% Vote Share — 2026", o50_26)


══════════════════════════════════════════════════════════════════════
  METRIC 18 — Candidates with >50% Vote Share
══════════════════════════════════════════════════════════════════════
  2021: 70 candidates crossed 50%
  2026: 13 candidates crossed 50%

══════════════════════════════════════════════════════════════════════
  METRIC 18 — All Candidates >50% Vote Share — 2021
══════════════════════════════════════════════════════════════════════
╭────────────────────────┬─────────────────────────┬─────────┬─────────┬────────────────╮
│ Constituency           │ Candidate               │ Party   │   Votes │   Vote Share % │
├────────────────────────┼─────────────────────────┼─────────┼─────────┼────────────────┤
│ Athoor                 │ PERIYASAMY I            │ DMK     │  165809 │          72.11 │
│ Chepauk-Thiruvallikeni │ UDHAYANIDHI STALIN      │ DMK     │   93285 │          67.89 │
│ Tiruvannamalai         │ E V VELU                │ DMK     │  137876 │          66.02 │
│ Edapad

In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# METRIC 19 — Winners with <35% Vote Share in their Constituency
# ══════════════════════════════════════════════════════════════════════════════
def winners_under35(winners_df, df_raw, year):
    # total_valid already computed in get_winners; use win_pct directly
    w = winners_df.copy()
    under = w[w['win_pct'] < 35]
    return under[['constituency','candidate','party','votes','win_pct']].rename(
        columns={'constituency':'Constituency','candidate':'Candidate',
                 'party':'Party','votes':'Votes','win_pct':'Winner Vote Share %'}).sort_values('Winner Vote Share %')

u35_26 = winners_under35(winners26, df26, 2026)
u35_21 = winners_under35(winners21, df21, 2021)
print(f"\n{'═'*70}")
print(f"  METRIC 19 — Winners with <35% Vote Share (fragmented contests)")
print(f"{'═'*70}")
print(f"  2021: {len(u35_21)} winners had <35% vote share")
print(f"  2026: {len(u35_26)} winners had <35% vote share")
pprint("METRIC 19 — Winners <35% Vote Share — 2021 (sample 20)", u35_21.head(20))
pprint("METRIC 19 — Winners <35% Vote Share — 2026 (sample 20)", u35_26.head(20))

print("\n\n✅ All metrics computed successfully.\n")


══════════════════════════════════════════════════════════════════════
  METRIC 19 — Winners with <35% Vote Share (fragmented contests)
══════════════════════════════════════════════════════════════════════
  2021: 2 winners had <35% vote share
  2026: 64 winners had <35% vote share

══════════════════════════════════════════════════════════════════════
  METRIC 19 — Winners <35% Vote Share — 2021 (sample 20)
══════════════════════════════════════════════════════════════════════
╭──────────────────┬────────────────────┬─────────┬─────────┬───────────────────────╮
│ Constituency     │ Candidate          │ Party   │   Votes │   Winner Vote Share % │
├──────────────────┼────────────────────┼─────────┼─────────┼───────────────────────┤
│ Usilampatti      │ AYYAPPAN P         │ AIADMK  │   71255 │                 33.53 │
│ Coimbatore South │ VANATHI SRINIVASAN │ BJP     │   53209 │                 34.38 │
╰──────────────────┴────────────────────┴─────────┴─────────┴───────────────────────╯